In [1]:
# packages
import pandas as pd
from mod02_build_bot_predictor import train_model

### Define a function to extract predictions from the model

In [3]:
def predict_bot(df, model=None):
    """
    Predict whether each account is a bot (1) or human (0).
    """
    if model is None:
        model = train_model()

    preds = model.predict(df)
    return pd.Series(preds, index=df.index)

### Define a function to evaluate model error

In [4]:
def confusion_matrix_and_metrics(y_true, y_pred):
    """
    Computes confusion matrix and common error rates for binary classification.

    Assumes labels:
      0 = negative class
      1 = positive class

    Returns:
      dict with:
        tn, fp, fn, tp
        misclassification_rate
        false_positive_rate
        false_negative_rate
    """
    tn = fp = fn = tp = 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 0 and yp == 0:
            tn += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
        elif yt == 1 and yp == 1:
            tp += 1
        else:
            raise ValueError("Labels must be 0 or 1")

    total = tn + fp + fn + tp

    misclassification_rate = (fp + fn) / total if total > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "misclassification_rate": misclassification_rate,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
    }


### Load the data

In [5]:
TRAIN_PATH = "mod02_data/train.csv"
train = pd.read_csv(TRAIN_PATH)

TEST_PATH = "mod02_data/test.csv"
test = pd.read_csv(TEST_PATH)

### Format the data by independent vs. dependent variables

In [6]:
X_train = train.drop(columns=["is_bot"])
y_train = train['is_bot']

X_test = test.drop(columns=["is_bot"])
y_test = test['is_bot']

### Build the model on training data

In [7]:
model = train_model(X_train, y_train)

### Get the model predictions on training and test data

In [8]:
y_pred_train = predict_bot(X_train, model)
y_pred_test = predict_bot(X_test, model)

### Check results on the training set (data used to build the model)

In [9]:
confusion_matrix_and_metrics(y_train, y_pred_train)

{'tp': 113,
 'tn': 2591,
 'fp': 46,
 'fn': 250,
 'misclassification_rate': 0.09866666666666667,
 'false_positive_rate': 0.017444065225635193,
 'false_negative_rate': 0.6887052341597796}

### Check results on the test set (new data not yet seen by the model)

In [10]:
confusion_matrix_and_metrics(y_test, y_pred_test)

{'tp': 33,
 'tn': 861,
 'fp': 13,
 'fn': 93,
 'misclassification_rate': 0.106,
 'false_positive_rate': 0.014874141876430207,
 'false_negative_rate': 0.7380952380952381}

# Discussion Questions

### Based on the misclassification rate of your model, discuss your confidence in the ability to predict a bot. 

With a test misclassification rate of about 10.6%, the model gets roughly 9 out of 10 predictions right overall, which sounds fine at first glance. But that number is a little misleading here because bots only make up about 12% of the dataset. Looking at the confusion matrix, the model is very good at recognizing humans (false positive rate of only ~1.5%), but it misses about 74% of actual bots (false negative rate). So overall I'm not very confident in this model's ability to reliably catch bots. It's mostly just predicting "human" most of the time and getting credit for the class imbalance, rather than having learned strong bot-specific patterns. The accuracy/misclassification number alone overstates how useful the model actually is for the thing we care about.

### What are potential ramifications of false positives from the model?

A false positive means a real human account gets flagged as a bot. Depending on how this model is deployed, that could mean a legitimate user gets rate-limited, locked out, shadow-banned, or has their content removed or deprioritized for no reason. That's a bad experience for the people affected, and it could disproportionately hit certain kinds of legitimate users, for example people who post frequently, use automation tools for legitimate purposes, or just have unusual-but-real behavior patterns. At scale, false positives like this erode user trust in the platform and can raise fairness concerns if they aren't spread evenly across different types of users.

### What are potential ramifications of false negatives from the model?

A false negative means an actual bot slips through undetected, which given this model's ~74% false negative rate is really the bigger issue. Undetected bots can be used to spread misinformation, inflate engagement metrics (likes, follows, trending topics), scrape data, or run spam/scam campaigns, all while blending in as normal activity. Because the model misses so many bots, I wouldn't trust it on its own as the main line of defense. It would need to be combined with other detection signals or used as one input into a broader system rather than the sole decision-maker.